In [ ]:
%run ./imports.py

In [ ]:
prefix_rtt_profiling_path = "data/prefix_rtt_profiling_path.pkl"
prefix_rtt_profiling_fixed_windows_path = "data/prefix_rtt_profiling_fixed_windows_path.pkl"
prefix_rtt_profiling_variable_windows_path = "data/prefix_rtt_profiling_variable_windows_path.pkl"
prefix_optimal_attacks_path = "data/prefix_optimal_attacks.pkl"
covered_prefixes_variable_windows_path = "data/covered_prefixes_variable_windows.pkl"
covered_prefixes_fixed_windows_path = "data/covered_prefixes_fixed_windows.pkl"

### Load optimal attacks data

In [ ]:
df_prefix_optimal_attacks = pd.read_pickle(prefix_optimal_attacks_path)
df_prefix_optimal_attacks.head(n=1)

In [ ]:
prefix_attacker_post_map = {}
prefix_attacker_dev_map = {}
for prefix, attacker, post, mindev in df_prefix_optimal_attacks[[
        "Prefix", "Attacker", "PostAttack_ms", "MinDev_ms"]].itertuples(index=False, name=None):
    if prefix not in prefix_attacker_post_map:
        prefix_attacker_post_map[prefix] = {}
    if prefix not in prefix_attacker_dev_map:
        prefix_attacker_dev_map[prefix] = {}
    if attacker not in prefix_attacker_post_map[prefix]:
        prefix_attacker_post_map[prefix][attacker] = post
    if attacker not in prefix_attacker_dev_map[prefix]:
        prefix_attacker_dev_map[prefix][attacker] = max(0, mindev)

## Variable windows-based coverage

In [ ]:
df_prefixes_profiling_variablewindows = pd.read_pickle(prefix_rtt_profiling_variable_windows_path)
df_prefixes_profiling_variablewindows.head(n=1)

In [ ]:
us_ciso_prefixes_varwin = df_prefixes_profiling_variablewindows[(df_prefixes_profiling_variablewindows[
                    "Connection_Type_Unique"].apply(lambda lst: len(lst)==1 and "CISO" in lst))
                    & (df_prefixes_profiling_variablewindows["Country"] == "United States of America")]["Destination_Prefix"].tolist()
us_ciso_prefixes_varwin = sorted(us_ciso_prefixes_varwin)
print(f"No. of US-based CISO prefixes: {len(us_ciso_prefixes_varwin)}")

In [ ]:
us_sico_prefixes_varwin = df_prefixes_profiling_variablewindows[(df_prefixes_profiling_variablewindows[
                    "Connection_Type_Unique"].apply(lambda lst: len(lst)==1 and "SICO" in lst))
                    & (df_prefixes_profiling_variablewindows["Country"] == "United States of America")]["Destination_Prefix"].tolist()
us_sico_prefixes_varwin = sorted(us_sico_prefixes_varwin)
print(f"No. of US-based SICO prefixes: {len(us_sico_prefixes_varwin)}")

### Absolute threshold-based coverage

In [ ]:
def compute_theory_coverage(mindev_map, prefixes):
    passing = {}
    coverages = []
    total_coverage = 0
    total_attacks = 0
    for prefix in prefixes:
        passing[prefix] = []
        mindev_prefix = [mindev_map[prefix][attacker] for attacker in mindev_map[prefix]]
        count = sum(1 for dev in mindev_prefix if dev >= 1)
        total_coverage += count
        total_attacks  += len(mindev_prefix)
        coverages.append(count * 100.0 / len(mindev_prefix))
        for attacker in mindev_map[prefix]:
            if mindev_map[prefix][attacker] >= 1:
                passing[prefix].append(attacker)
    return coverages, passing, (total_coverage * 100.0 / total_attacks)

In [ ]:
coverages_ciso_varwin, passing_ciso_varwin, total_coverage_ciso_varwin = compute_theory_coverage(prefix_attacker_dev_map, us_ciso_prefixes_varwin)
coverages_sico_varwin, passing_sico_varwin, total_coverage_sico_varwin = compute_theory_coverage(prefix_attacker_dev_map, us_sico_prefixes_varwin)
coverages_ciso_varwin = sorted(coverages_ciso_varwin, reverse=True)
coverages_sico_varwin = sorted(coverages_sico_varwin, reverse=True)
print("Total coverage:")
print(f"\tCISO: {mu.rnd(total_coverage_ciso_varwin, 1)}%")
print(f"\tSICO: {mu.rnd(total_coverage_sico_varwin, 1)}%")

In [ ]:
pct_prefixes_varwin = {"ciso": [], "sico": []}
pct_attacks_varwin  = {"ciso": [], "sico": []}

for ptype in pct_prefixes_varwin:
    if ptype == "ciso":
        coverages_ptype = coverages_ciso_varwin.copy()
    else:
        coverages_ptype = coverages_sico_varwin.copy()
    for c in range(1, len(coverages_ptype)+1):
        a = coverages_ptype[c-1]
        pct_prefixes_varwin[ptype].append(c * 100.0 / len(coverages_ptype))
        pct_attacks_varwin[ptype].append(a)

In [ ]:
sns_colors = list(sns.color_palette("bright"))
pu.lineplots(
    [pct_prefixes_varwin["ciso"], pct_prefixes_varwin["sico"]],
    [pct_attacks_varwin["ciso"], pct_attacks_varwin["sico"]], {
    "figsize": (8, 6),
    "colors": sns_colors,
    "linestyles": ["-", "--"],
    "loc": "lower left",
    "curvelabels": ["Defend Campus Clients", "Defend Campus Servers"],
    "xlim": (-5, 105),
    "ylim": (-5, 105),
    "xlabel": "US Prefixes (%)",
    "ylabel": "Non-US Attacks (%)",
    "plot_path": "plots/prefixes_vs_attacks_theory_varwin.pdf"
})

### Surge threshold-based coverage

In [ ]:
prefix_maxmin_varwin_map = df_prefixes_profiling_variablewindows.set_index(
    'Destination_Prefix')["Profiling_Variable_Max_Min_RTT_per_Window_ms"].to_dict()

In [ ]:
def compute_surge_coverage(prefixes, midrtt_map, mindev_map, maxmin_map, surges):
    passing   = {}
    coverages = {}
    total_coverage = {}
    total_attacks  = {}
    
    for surge in surges:
        passing[surge]   = {}
        coverages[surge] = []
        total_coverage[surge] = 0
        total_attacks[surge]  = 0

    for surge in surges:
        for prefix in prefixes:
            passing[surge][prefix] = []
            max_min = maxmin_map[prefix]
            count_passing = 0
            
            for attacker in midrtt_map[prefix]:
                mid_rtt = midrtt_map[prefix][attacker]
                min_dev = mindev_map[prefix][attacker]
                
                if min_dev >= 1 and mid_rtt - max_min > surge:
                    passing[surge][prefix].append(attacker)
                    count_passing += 1

            total_coverage[surge] += count_passing
            total_attacks[surge]  += len(midrtt_map[prefix])
            coverages[surge].append(count_passing * 100.0 / len(midrtt_map[prefix]))
            
    return coverages, passing, total_coverage, total_attacks

In [ ]:
surge_thresholds = list(range(5, 101, 5))

coverages_surge_ciso_varwin, passing_surge_ciso_varwin, total_coverage_surge_ciso_varwin, total_attacks_surge_ciso_varwin = compute_surge_coverage(
    us_ciso_prefixes_varwin, prefix_attacker_post_map, prefix_attacker_dev_map, prefix_maxmin_varwin_map, surge_thresholds)
coverages_surge_sico_varwin, passing_surge_sico_varwin, total_coverage_surge_sico_varwin, total_attacks_surge_sico_varwin = compute_surge_coverage(
    us_sico_prefixes_varwin, prefix_attacker_post_map, prefix_attacker_dev_map, prefix_maxmin_varwin_map, surge_thresholds)

for surge in surge_thresholds:
    coverages_surge_ciso_varwin[surge] = sorted(coverages_surge_ciso_varwin[surge], reverse=True)
    coverages_surge_sico_varwin[surge] = sorted(coverages_surge_sico_varwin[surge], reverse=True)
    print(f"Surge threshold: {surge} ms")
    ciso_cov = total_coverage_surge_ciso_varwin[surge] * 100.0 / total_attacks_surge_ciso_varwin[surge]
    sico_cov = total_coverage_surge_sico_varwin[surge] * 100.0 / total_attacks_surge_sico_varwin[surge]
    print(f"\tCoverage for CISO: {mu.rnd(ciso_cov, 1)}%")
    print(f"\tCoverage for SICO: {mu.rnd(sico_cov, 1)}%")

In [ ]:
pct_prefixes_surge_varwin = {"ciso": {}, "sico": {}}
pct_attacks_surge_varwin  = {"ciso": {}, "sico": {}}

for sth in surge_thresholds:
    pct_prefixes_surge_varwin["ciso"][sth] = []
    pct_prefixes_surge_varwin["sico"][sth] = []
    pct_attacks_surge_varwin["ciso"][sth]  = []
    pct_attacks_surge_varwin["sico"][sth]  = []

for ptype in pct_prefixes_surge_varwin:
    for surge in pct_prefixes_surge_varwin[ptype]:
        if ptype == "ciso":
            coverages_ptype = coverages_surge_ciso_varwin[surge].copy()
        else:
            coverages_ptype = coverages_surge_sico_varwin[surge].copy()
        for c in range(1, len(coverages_ptype)+1):
            a = coverages_ptype[c-1]
            pct_prefixes_surge_varwin[ptype][surge].append(c * 100.0 / len(coverages_ptype))
            pct_attacks_surge_varwin[ptype][surge].append(a)

In [ ]:
sns_colors = list(sns.color_palette("bright"))
pu.lineplots(
    [pct_prefixes_surge_varwin["ciso"][5], pct_prefixes_surge_varwin["ciso"][20],
     pct_prefixes_surge_varwin["sico"][5], pct_prefixes_surge_varwin["sico"][20]],
    [pct_attacks_surge_varwin["ciso"][5],  pct_attacks_surge_varwin["ciso"][20],
     pct_attacks_surge_varwin["sico"][5],  pct_attacks_surge_varwin["sico"][20]],
    {
        "figsize": (8, 6),
        "colors": [sns_colors[0], sns_colors[0], sns_colors[1], sns_colors[1]],
        "linestyles": ["-", "--"],
        "loc": "lower left",
        "curvelabels": ["Clients (5 ms surge)", "Clients (20 ms surge)", "Servers (5 ms surge)", "Servers (20 ms surge)"],
        "xlim": (-5, 105),
        "ylim": (-5, 105),
        "xlabel": "US Prefixes (%)",
        "ylabel": "Non-US Attacks (%)",
        "framealpha": 0.5,
        "plot_path": "plots/prefixes_vs_attacks_surge_varwin.pdf"
})

In [ ]:
with open(covered_prefixes_variable_windows_path, "wb") as fp:
    pickle.dump({
        "ciso": {
            "coverages": coverages_surge_ciso_varwin,
            "passing": passing_surge_ciso_varwin,
            "total_coverage": total_coverage_surge_ciso_varwin,
            "total_attacks": total_attacks_surge_ciso_varwin
        },
        "sico": {
            "coverages": coverages_surge_sico_varwin,
            "passing": passing_surge_sico_varwin,
            "total_coverage": total_coverage_surge_sico_varwin,
            "total_attacks": total_attacks_surge_sico_varwin
        }
    }, fp)

## Fixed windows-based coverage

In [ ]:
df_prefixes_profiling_fixedwindows = pd.read_pickle(prefix_rtt_profiling_fixed_windows_path)
df_prefixes_profiling_fixedwindows.head(n=1)

In [ ]:
us_ciso_prefixes_fxdwin = df_prefixes_profiling_fixedwindows[(df_prefixes_profiling_fixedwindows[
                    "Connection_Type_Unique"].apply(lambda lst: len(lst)==1 and "CISO" in lst))
                    & (df_prefixes_profiling_fixedwindows["Country"] == "United States of America")]["Destination_Prefix"].tolist()
us_ciso_prefixes_fxdwin = sorted(us_ciso_prefixes_fxdwin)
print(f"No. of US-based CISO prefixes: {len(us_ciso_prefixes_fxdwin)}")

In [ ]:
us_sico_prefixes_fxdwin = df_prefixes_profiling_fixedwindows[(df_prefixes_profiling_fixedwindows[
                    "Connection_Type_Unique"].apply(lambda lst: len(lst)==1 and "SICO" in lst))
                    & (df_prefixes_profiling_fixedwindows["Country"] == "United States of America")]["Destination_Prefix"].tolist()
us_sico_prefixes_fxdwin = sorted(us_sico_prefixes_fxdwin)
print(f"No. of US-based SICO prefixes: {len(us_sico_prefixes_fxdwin)}")

### Absolute threshold-based coverage

In [ ]:
coverages_ciso_fxdwin, passing_ciso_fxdwin, total_coverage_ciso_fxdwin = compute_theory_coverage(prefix_attacker_dev_map, us_ciso_prefixes_fxdwin)
coverages_sico_fxdwin, passing_sico_fxdwin, total_coverage_sico_fxdwin = compute_theory_coverage(prefix_attacker_dev_map, us_sico_prefixes_fxdwin)
coverages_ciso_fxdwin = sorted(coverages_ciso_fxdwin, reverse=True)
coverages_sico_fxdwin = sorted(coverages_sico_fxdwin, reverse=True)
print("Total coverage:")
print(f"\tCISO: {mu.rnd(total_coverage_ciso_fxdwin, 1)}%")
print(f"\tSICO: {mu.rnd(total_coverage_sico_fxdwin, 1)}%")

In [ ]:
pct_prefixes_fxdwin = {"ciso": [], "sico": []}
pct_attacks_fxdwin  = {"ciso": [], "sico": []}

for ptype in pct_prefixes_fxdwin:
    if ptype == "ciso":
        coverages_ptype = coverages_ciso_fxdwin.copy()
    else:
        coverages_ptype = coverages_sico_fxdwin.copy()
    for c in range(1, len(coverages_ptype)+1):
        a = coverages_ptype[c-1]
        pct_prefixes_fxdwin[ptype].append(c * 100.0 / len(coverages_ptype))
        pct_attacks_fxdwin[ptype].append(a)

In [ ]:
sns_colors = list(sns.color_palette("bright"))
pu.lineplots(
    [pct_prefixes_fxdwin["ciso"], pct_prefixes_fxdwin["sico"]],
    [pct_attacks_fxdwin["ciso"], pct_attacks_fxdwin["sico"]], {
    "figsize": (8, 6),
    "colors": sns_colors,
    "linestyles": ["-", "--"],
    "loc": "lower left",
    "curvelabels": ["Defend Campus Clients", "Defend Campus Servers"],
    "xlim": (-5, 105),
    "ylim": (-5, 105),
    "xlabel": "US Prefixes (%)",
    "ylabel": "Non-US Attacks (%)",
    "plot_path": "plots/prefixes_vs_attacks_theory_fxdwin.pdf"
})

### Surge threshold-based coverage

In [ ]:
prefix_maxmin_fxdwin_map = df_prefixes_profiling_fixedwindows.set_index(
    'Destination_Prefix')["Profiling_Fixed_Max_Min_RTT_per_Window_ms"].to_dict()

In [ ]:
surge_thresholds = list(range(5, 26, 5))

coverages_surge_ciso_fxdwin, passing_surge_ciso_fxdwin, total_coverage_surge_ciso_fxdwin, total_attacks_surge_ciso_fxdwin = compute_surge_coverage(
    us_ciso_prefixes_fxdwin, prefix_attacker_post_map, prefix_attacker_dev_map, prefix_maxmin_fxdwin_map, surge_thresholds)
coverages_surge_sico_fxdwin, passing_surge_sico_fxdwin, total_coverage_surge_sico_fxdwin, total_attacks_surge_sico_fxdwin = compute_surge_coverage(
    us_sico_prefixes_fxdwin, prefix_attacker_post_map, prefix_attacker_dev_map, prefix_maxmin_fxdwin_map, surge_thresholds)

for surge in surge_thresholds:
    coverages_surge_ciso_fxdwin[surge] = sorted(coverages_surge_ciso_fxdwin[surge], reverse=True)
    coverages_surge_sico_fxdwin[surge] = sorted(coverages_surge_sico_fxdwin[surge], reverse=True)
    print(f"Surge threshold: {surge} ms")
    ciso_cov = total_coverage_surge_ciso_fxdwin[surge] * 100.0 / total_attacks_surge_ciso_fxdwin[surge]
    sico_cov = total_coverage_surge_sico_fxdwin[surge] * 100.0 / total_attacks_surge_sico_fxdwin[surge]
    print(f"\tCoverage for CISO: {mu.rnd(ciso_cov, 1)}%")
    print(f"\tCoverage for SICO: {mu.rnd(sico_cov, 1)}%")

In [ ]:
pct_prefixes_surge_fxdwin = {"ciso": {5: [], 20: []}, "sico": {5: [], 20: []}}
pct_attacks_surge_fxdwin  = {"ciso": {5: [], 20: []}, "sico": {5: [], 20: []}}

for ptype in pct_prefixes_surge_fxdwin:
    for surge in pct_prefixes_surge_fxdwin[ptype]:
        if ptype == "ciso":
            coverages_ptype = coverages_surge_ciso_fxdwin[surge].copy()
        else:
            coverages_ptype = coverages_surge_sico_fxdwin[surge].copy()
        for c in range(1, len(coverages_ptype)+1):
            a = coverages_ptype[c-1]
            pct_prefixes_surge_fxdwin[ptype][surge].append(c * 100.0 / len(coverages_ptype))
            pct_attacks_surge_fxdwin[ptype][surge].append(a)

In [ ]:
sns_colors = list(sns.color_palette("bright"))
pu.lineplots(
    [pct_prefixes_surge_fxdwin["ciso"][5], pct_prefixes_surge_fxdwin["ciso"][20],
     pct_prefixes_surge_fxdwin["sico"][5], pct_prefixes_surge_fxdwin["sico"][20]],
    [pct_attacks_surge_fxdwin["ciso"][5], pct_attacks_surge_fxdwin["ciso"][20],
     pct_attacks_surge_fxdwin["sico"][5], pct_attacks_surge_fxdwin["sico"][20]],
    
    {
        "figsize": (8, 6),
        "colors": [sns_colors[0], sns_colors[0], sns_colors[1], sns_colors[1]],
        "linestyles": ["-", "--"],
        "loc": "lower left",
        "curvelabels": ["Clients (5 ms surge)", "Clients (20 ms surge)", "Servers (5 ms surge)", "Servers (20 ms surge)"],
        "xlim": (-5, 105),
        "ylim": (-5, 105),
        "xlabel": "US Prefixes (%)",
        "ylabel": "Non-US Attacks (%)",
        "framealpha": 0.5,
        "plot_path": "plots/prefixes_vs_attacks_surge_fxdwin.pdf"
})

In [ ]:
with open(covered_prefixes_fixed_windows_path, "wb") as fp:
    pickle.dump({
        "ciso": {
            "coverages": coverages_surge_ciso_fxdwin,
            "passing": passing_surge_ciso_fxdwin,
            "total_coverage": total_coverage_surge_ciso_fxdwin,
            "total_attacks": total_attacks_surge_ciso_fxdwin
        },
        "sico": {
            "coverages": coverages_surge_sico_fxdwin,
            "passing": passing_surge_sico_fxdwin,
            "total_coverage": total_coverage_surge_sico_fxdwin,
            "total_attacks": total_attacks_surge_sico_fxdwin
        }
    }, fp)